In [1]:
# Celda 1 – Librerías  [V0.2]
import cv2
import numpy as np
import time
import customtkinter as ctk
from PIL import Image
import datetime

ctk.set_appearance_mode("dark")
print("Celda 1 V0.2: Librerías cargadas correctamente.")

Celda 1 V0.2: Librerías cargadas correctamente.


In [2]:
# ─── Celda 2 – Motor de visión  [V0.2] ───────────────────────────────────────
from pygrabber.dshow_graph import FilterGraph


def detectar_camaras_sistema():
    """Lee el registro de Windows sin encender el hardware."""
    try:
        graph = FilterGraph()
        return [(i, f'📷 {n}') for i, n in enumerate(graph.get_input_devices())]
    except Exception as e:
        print(f'Error al buscar cámaras: {e}')
        return []


# ─── Rangos HSV (H: 0-179, S: 0-255, V: 0-255) ───────────────────────────────
RANGO_ROJO_1 = (np.array([  0, 150,  80]), np.array([  8, 255, 255]))
RANGO_ROJO_2 = (np.array([172, 150,  80]), np.array([180, 255, 255]))
RANGO_VERDE  = (np.array([ 45, 140,  80]), np.array([ 78, 255, 255]))
RANGO_AZUL   = (np.array([108, 150,  60]), np.array([128, 255, 255]))

RATIO_COB_MIN = 0.25
FRAC_AREA_MIN = 0.005

KERNEL_OPEN  = np.ones((5, 5), np.uint8)
KERNEL_CLOSE = np.ones((7, 7), np.uint8)
KERNEL_ERODE = np.ones((3, 3), np.uint8)

# Umbral de similitud de histograma (0-1).
# > UMBRAL  → mismo objeto ya registrado (no duplicar)
UMBRAL_MISMO_OBJETO = 0.82

# Segundos de escena vacía necesarios para el modo Reinicio
TIEMPO_REINICIO_SEG = 7.0


# ─── Huella digital por histograma HSV ───────────────────────────────────────
def calcular_huella_hsv(frame_hsv, mascara, bbox):
    """
    Histograma 2D H x S sobre los píxeles enmascarados del bbox.
    Captura tono + saturación = 'firma' del color del objeto.
    Retorna histograma normalizado, o None si hay pocos píxeles.
    """
    x, y, w, h  = bbox
    roi_hsv     = frame_hsv[y:y+h, x:x+w]
    roi_mask    = mascara[y:y+h, x:x+w]
    if cv2.countNonZero(roi_mask) < 50:
        return None
    hist = cv2.calcHist(
        [roi_hsv], [0, 1], roi_mask,
        [30, 32], [0, 180, 0, 256]
    )
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return hist


def es_mismo_objeto(huella_nueva, huellas_guardadas, umbral=UMBRAL_MISMO_OBJETO):
    """True si la huella_nueva supera el umbral de similitud con alguna guardada."""
    if huella_nueva is None or not huellas_guardadas:
        return False
    for ref in huellas_guardadas:
        if cv2.compareHist(huella_nueva, ref, cv2.HISTCMP_CORREL) >= umbral:
            return True
    return False


# ─── TrackerColor ─────────────────────────────────────────────────────────────
class TrackerColor:
    """Estabiliza el bounding-box con EMA adaptativo y zona muerta."""

    def __init__(
        self,
        alpha_min=0.30, alpha_max=0.75,
        umbral_px=8,    dist_max_px=80,
        frames_conf=2,  frames_perdida=4,
    ):
        self.alpha_min      = alpha_min
        self.alpha_max      = alpha_max
        self.umbral_px      = umbral_px
        self.dist_max_px    = dist_max_px
        self.frames_conf    = frames_conf
        self.frames_perdida = frames_perdida
        self.reiniciar()

    def reiniciar(self):
        self.bbox_suave  = None
        self.conteo_det  = 0
        self.conteo_perd = 0
        self.visible     = False

    def actualizar(self, bbox_raw, frame_shape):
        h_f, w_f = frame_shape[:2]
        escala   = max(h_f, w_f) / 640.0
        umbral   = self.umbral_px   * escala
        dist_max = self.dist_max_px * escala

        if bbox_raw is None:
            self.conteo_det  = 0
            self.conteo_perd = min(self.conteo_perd + 1, self.frames_perdida + 1)
            if self.conteo_perd >= self.frames_perdida:
                self.visible    = False
                self.bbox_suave = None
            return self._int() if self.visible else None

        self.conteo_perd = 0
        self.conteo_det  = min(self.conteo_det + 1, self.frames_conf + 10)
        if self.conteo_det < self.frames_conf:
            return None
        if self.bbox_suave is None:
            self.bbox_suave = [float(v) for v in bbox_raw]
            self.visible    = True
            return self._int()

        self.visible = True
        cx_r = bbox_raw[0] + bbox_raw[2] / 2.0
        cy_r = bbox_raw[1] + bbox_raw[3] / 2.0
        cx_s = self.bbox_suave[0] + self.bbox_suave[2] / 2.0
        cy_s = self.bbox_suave[1] + self.bbox_suave[3] / 2.0
        dist = ((cx_r - cx_s)**2 + (cy_r - cy_s)**2)**0.5

        if dist < umbral:
            a = self.alpha_min * 0.4
            self.bbox_suave[2] = a * bbox_raw[2] + (1-a) * self.bbox_suave[2]
            self.bbox_suave[3] = a * bbox_raw[3] + (1-a) * self.bbox_suave[3]
        else:
            t     = min(1.0, dist / dist_max)
            alpha = self.alpha_min + t * (self.alpha_max - self.alpha_min)
            for i in range(4):
                self.bbox_suave[i] = alpha * bbox_raw[i] + (1-alpha) * self.bbox_suave[i]

        return self._int()

    def _int(self):
        if self.bbox_suave is None:
            return None
        return tuple(int(round(v)) for v in self.bbox_suave)


# ─── Helpers ──────────────────────────────────────────────────────────────────
def calcular_orientacion(w, h):
    if h == 0 or w == 0:
        return 'Desconocido', 1.0
    ratio = h / w
    if   ratio > 1.3:  return 'Vertical',   ratio
    elif ratio < 0.77: return 'Horizontal',  ratio
    else:              return 'Cuadrado',    ratio


def estimar_z(area_px, frame_shape):
    frac = area_px / (frame_shape[0] * frame_shape[1])
    return round(max(0.1, min(5.0, 1.0 / (frac * 10 + 0.01))), 2)


# ─── Procesamiento de frame ───────────────────────────────────────────────────
def procesar_frame_vision(frame, temporizadores, trackers, callback_objeto=None):
    """
    Detecta Rojo/Verde/Azul con contornos precisos.
    Retorna (frame_procesado, hay_objetos: bool)
    callback_objeto(nombre, orientacion, x_norm, y_norm, z_est, huella)
    """
    blurred = cv2.GaussianBlur(frame, (5, 5), 0)
    hsv     = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    t_act   = time.time()
    area_min = max(2000, int(frame.shape[0] * frame.shape[1] * FRAC_AREA_MIN))

    m_rojo  = cv2.add(cv2.inRange(hsv, *RANGO_ROJO_1),
                      cv2.inRange(hsv, *RANGO_ROJO_2))
    m_verde = cv2.inRange(hsv, *RANGO_VERDE)
    m_azul  = cv2.inRange(hsv, *RANGO_AZUL)

    colores = [
        ('Rojo',  m_rojo,  (0,   0, 255)),
        ('Verde', m_verde, (0, 255,   0)),
        ('Azul',  m_azul,  (255, 0,   0)),
    ]

    hay_objetos = False

    for nombre, mascara, color_bgr in colores:
        mascara  = cv2.morphologyEx(mascara, cv2.MORPH_OPEN,  KERNEL_OPEN)
        mascara  = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, KERNEL_CLOSE)
        m_ref    = cv2.erode(mascara, KERNEL_ERODE, iterations=1)
        m_ref    = cv2.dilate(m_ref,  KERNEL_ERODE, iterations=2)

        contornos, _ = cv2.findContours(
            m_ref, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_KCOS
        )

        bbox_raw = None
        c_valido = None
        if contornos:
            c    = max(contornos, key=cv2.contourArea)
            area = cv2.contourArea(c)
            if area >= area_min:
                x, y, w, h = cv2.boundingRect(c)
                px_c = cv2.countNonZero(mascara[y:y+h, x:x+w])
                if px_c / (w * h) >= RATIO_COB_MIN:
                    bbox_raw = (x, y, w, h)
                    c_valido = c

        bbox_e = trackers[nombre].actualizar(bbox_raw, frame.shape)

        if bbox_e is None:
            temporizadores[nombre] = 0.0
            continue

        hay_objetos = True
        x, y, w, h  = bbox_e

        if temporizadores[nombre] == 0.0:
            temporizadores[nombre] = t_act

        label_y    = max(18, y - 12)
        orient, _  = calcular_orientacion(w, h)
        fh, fw     = frame.shape[:2]
        x_norm     = round((x + w/2.0)/fw * 2 - 1, 2)
        y_norm     = round(1 - (y + h/2.0)/fh * 2, 2)
        z_est      = estimar_z(w * h, frame.shape)

        en_calib = (t_act - temporizadores[nombre]) <= 3.0

        if en_calib:
            paso = 15
            for i in range(x, x + w, paso):
                cv2.line(frame, (i, y), (i, y+h), color_bgr, 1)
            for j in range(y, y + h, paso):
                cv2.line(frame, (x, j), (x+w, j), color_bgr, 1)
            cv2.putText(frame, 'Calibrando...', (x, label_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)
        else:
            if c_valido is not None:
                eps  = 0.008 * cv2.arcLength(c_valido, True)
                c_s  = cv2.approxPolyDP(c_valido, eps, True)
                cv2.drawContours(frame, [c_s], -1, color_bgr, 2)
                cv2.rectangle(frame, (x, y), (x+w, y+h), color_bgr, 1)
            else:
                cv2.rectangle(frame, (x, y), (x+w, y+h), color_bgr, 2)

            arrow  = '↕' if orient == 'Vertical' else ('↔' if orient == 'Horizontal' else '⊙')
            lbl_p  = f'{nombre} {arrow}'
            lbl_c  = f'X:{x_norm:+.1f} Y:{y_norm:+.1f} Z:{z_est}m'

            (tw, th), _ = cv2.getTextSize(lbl_p, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(frame, (x, label_y-th-4), (x+tw+4, label_y+4), (30,30,30), -1)
            cv2.putText(frame, lbl_p, (x+2, label_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color_bgr, 2)

            cy2 = label_y + th + 6
            (cw, ch), _ = cv2.getTextSize(lbl_c, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
            cv2.rectangle(frame, (x, cy2-ch-2), (x+cw+4, cy2+2), (30,30,30), -1)
            cv2.putText(frame, lbl_c, (x+2, cy2),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.42, (220,220,220), 1)

            if callback_objeto is not None:
                huella = calcular_huella_hsv(hsv, mascara, (x, y, w, h))
                callback_objeto(nombre, orient, x_norm, y_norm, z_est, huella)

    return frame, hay_objetos


print('Celda 2 V0.2: Motor de visión compilado.')

Celda 2 V0.2: Motor de visión compilado.


In [3]:
# ─── Celda 3 – UI  [V0.2] ────────────────────────────────────────────────────

# Estados del modo Reinicio
REINICIO_INACTIVO  = 0   # normal
REINICIO_ESPERANDO = 1   # esperando que la escena quede vacía
REINICIO_LISTO     = 2   # escena vacía confirmada, memoria borrada


class EscanerApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title('Sistema de Detección Profesional  —  V0.2')
        self.geometry('1380x860')
        self.configure(fg_color='#1a1d2e')

        self.cap            = None
        self.escaneando     = False
        self.temporizadores = {'Rojo': 0.0, 'Verde': 0.0, 'Azul': 0.0}

        self.trackers = {
            'Rojo':  TrackerColor(),
            'Verde': TrackerColor(),
            'Azul':  TrackerColor(),
        }

        # ── Memoria de huellas por color ──────────────────────────────────────
        self.huellas_memoria   = {'Rojo': [], 'Verde': [], 'Azul': []}
        self.historial_objetos = []
        self.ultimo_intento    = {}
        self.COOLDOWN_INTENTO  = 1.5

        # ── Estado del modo Reinicio ──────────────────────────────────────────
        self.modo_reinicio       = REINICIO_INACTIVO
        self.t_escena_vacia      = None    # timestamp de cuándo la escena quedó limpia
        self.frames_sin_objeto   = 0       # contador de frames sin ningún objeto
        self.FRAMES_VACIA_MIN    = 8       # frames consecutivos sin nada para iniciar cuenta

        self._construir_encabezado()
        self._construir_panel_config()
        self._construir_panel_video()

    # =========================================================================
    # CONSTRUCCIÓN DE LA UI
    # =========================================================================

    def _construir_encabezado(self):
        self.lbl_titulo = ctk.CTkLabel(
            self, text='CONFIGURACIÓN DE CÁMARA',
            font=('Helvetica', 24, 'bold'), text_color='#e8eaf0'
        )
        self.lbl_titulo.pack(pady=(28, 12))

    def _construir_panel_config(self):
        self.btn_detectar = ctk.CTkButton(
            self, text='🔍 DETECTAR CÁMARAS',
            fg_color='#3498db', hover_color='#2980b9',
            font=('Helvetica', 14, 'bold'), command=self._accion_buscar
        )
        self.btn_detectar.pack(pady=10)

        self.lbl_estado = ctk.CTkLabel(
            self, text='Haz clic para buscar dispositivos',
            text_color='#a0aab5', font=('Helvetica', 12)
        )
        self.lbl_estado.pack(pady=(0, 20))

        self.lbl_lista_tit = ctk.CTkLabel(
            self, text='Cámaras Disponibles:',
            font=('Helvetica', 14, 'bold'), text_color='#ffffff'
        )
        self.lbl_lista_tit.pack(anchor='w', padx=150)

        self.frame_lista = ctk.CTkScrollableFrame(
            self, fg_color='#2e3548', width=700, height=150,
            corner_radius=8, border_width=1, border_color='#4a5568'
        )
        self.frame_lista.pack(pady=5, padx=150)

        self.indice_sel = ctk.StringVar(value='-1')

        self.lbl_pie = ctk.CTkLabel(
            self, text='Ninguna cámara seleccionada',
            text_color='#a0aab5', font=('Helvetica', 12)
        )
        self.lbl_pie.pack(anchor='w', padx=150)

        self.btn_iniciar = ctk.CTkButton(
            self, text='INICIAR DETECCIÓN',
            fg_color='#3b8ed0', hover_color='#296899',
            font=('Helvetica', 16, 'bold'), width=250, height=45,
            state='disabled', command=self._iniciar_camara
        )
        self.btn_iniciar.pack(pady=(40, 0))

    def _construir_panel_video(self):
        # ── Contenedor horizontal principal ───────────────────────────────────
        self.frame_monitor = ctk.CTkFrame(self, fg_color='transparent')

        # ── Columna izquierda: video + controles ──────────────────────────────
        self.frame_video_col = ctk.CTkFrame(self.frame_monitor, fg_color='transparent')
        self.frame_video_col.pack(side='left', fill='both', expand=True)

        self.lbl_video = ctk.CTkLabel(self.frame_video_col, text='')
        self.lbl_video.pack(pady=10, padx=10)

        # ── Barra de estado del Reinicio ──────────────────────────────────────
        self.frame_reinicio_barra = ctk.CTkFrame(
            self.frame_video_col, fg_color='#0d1117',
            corner_radius=8, height=46
        )
        self.frame_reinicio_barra.pack(fill='x', padx=12, pady=(0, 4))
        self.frame_reinicio_barra.pack_propagate(False)

        self.lbl_reinicio_estado = ctk.CTkLabel(
            self.frame_reinicio_barra,
            text='',
            font=('Helvetica', 11, 'bold'),
            text_color='#556070'
        )
        self.lbl_reinicio_estado.pack(side='left', padx=12)

        self.barra_progreso = ctk.CTkProgressBar(
            self.frame_reinicio_barra,
            width=320, height=12,
            fg_color='#1e2535',
            progress_color='#f39c12'
        )
        self.barra_progreso.set(0)
        self.barra_progreso.pack(side='right', padx=12)

        # ── Fila de botones de control ────────────────────────────────────────
        self.frame_controles = ctk.CTkFrame(self.frame_video_col, fg_color='transparent')
        self.frame_controles.pack(pady=8)

        self.btn_escaneo = ctk.CTkButton(
            self.frame_controles, text='▶ Iniciar Escaneo',
            font=('Helvetica', 13, 'bold'),
            fg_color='#f39c12', hover_color='#d68910',
            command=self._toggle_escaneo
        )
        self.btn_escaneo.grid(row=0, column=0, padx=10)

        self.btn_reinicio = ctk.CTkButton(
            self.frame_controles,
            text='🔄 Reinicio',
            font=('Helvetica', 13, 'bold'),
            fg_color='#8e44ad', hover_color='#6c3483',
            width=140,
            command=self._iniciar_modo_reinicio,
            state='disabled'
        )
        self.btn_reinicio.grid(row=0, column=1, padx=10)

        self.btn_limpiar_todo = ctk.CTkButton(
            self.frame_controles,
            text='🗑 Limpiar todo',
            font=('Helvetica', 13, 'bold'),
            fg_color='#c0392b', hover_color='#96281b',
            width=140,
            command=self._limpiar_todo
        )
        self.btn_limpiar_todo.grid(row=0, column=2, padx=10)

        self.btn_detener = ctk.CTkButton(
            self.frame_controles, text='⏹ Detener Cámara',
            font=('Helvetica', 13, 'bold'),
            fg_color='#e74c3c', hover_color='#c0392b',
            command=self._apagar_hardware
        )
        self.btn_detener.grid(row=0, column=3, padx=10)

        # ── Columna derecha: historial ─────────────────────────────────────────
        self.frame_historial = ctk.CTkFrame(
            self.frame_monitor,
            fg_color='#20253a',
            width=300,
            corner_radius=12,
            border_width=1,
            border_color='#343d5c'
        )
        self.frame_historial.pack(side='right', fill='y', padx=(0, 12), pady=10)
        self.frame_historial.pack_propagate(False)

        encab = ctk.CTkFrame(self.frame_historial, fg_color='#151a2d', corner_radius=10)
        encab.pack(fill='x', padx=8, pady=(8, 4))

        ctk.CTkLabel(
            encab, text='📋 OBJETOS DETECTADOS',
            font=('Helvetica', 12, 'bold'), text_color='#c8d0e0'
        ).pack(pady=(8, 2))

        self.lbl_conteo = ctk.CTkLabel(
            encab, text='Total: 0  |  En memoria: 0',
            font=('Helvetica', 9), text_color='#7a8aaa'
        )
        self.lbl_conteo.pack(pady=(0, 8))

        self.lista_scroll = ctk.CTkScrollableFrame(
            self.frame_historial, fg_color='#1a1f33', corner_radius=8
        )
        self.lista_scroll.pack(fill='both', expand=True, padx=8, pady=(0, 8))

    # =========================================================================
    # LÓGICA DE REINICIO INTELIGENTE
    # =========================================================================

    def _iniciar_modo_reinicio(self):
        """
        Activa el modo Reinicio:
        - El sistema espera a que la escena quede completamente vacía.
        - Una vez vacía por TIEMPO_REINICIO_SEG segundos, borra solo
          las huellas de memoria → el mismo objeto será aceptado de nuevo
          como una instancia nueva (versión 2, 3…)
        """
        if self.modo_reinicio != REINICIO_INACTIVO:
            # Cancelar reinicio activo
            self._cancelar_reinicio()
            return

        self.modo_reinicio     = REINICIO_ESPERANDO
        self.t_escena_vacia    = None
        self.frames_sin_objeto = 0
        self.barra_progreso.set(0)

        self.btn_reinicio.configure(
            text='❌ Cancelar reinicio',
            fg_color='#6c3483', hover_color='#4a235a'
        )
        self._actualizar_barra_reinicio()

    def _cancelar_reinicio(self):
        self.modo_reinicio     = REINICIO_INACTIVO
        self.t_escena_vacia    = None
        self.frames_sin_objeto = 0
        self.barra_progreso.set(0)
        self.btn_reinicio.configure(
            text='🔄 Reinicio',
            fg_color='#8e44ad', hover_color='#6c3483'
        )
        self.lbl_reinicio_estado.configure(
            text='Reinicio cancelado.', text_color='#e74c3c'
        )
        self.after(2000, lambda: self.lbl_reinicio_estado.configure(text=''))

    def _procesar_estado_reinicio(self, hay_objetos: bool):
        """
        Llamado cada frame mientras escaneando. Maneja la máquina de estados
        del Reinicio.
        """
        if self.modo_reinicio == REINICIO_INACTIVO:
            return

        if self.modo_reinicio == REINICIO_ESPERANDO:
            if hay_objetos:
                # Objeto detectado → reiniciar el contador de escena vacía
                self.frames_sin_objeto = 0
                self.t_escena_vacia    = None
                self.barra_progreso.set(0)
                self.lbl_reinicio_estado.configure(
                    text='⚠ Retira todos los objetos del campo de visión…',
                    text_color='#e67e22'
                )
            else:
                # Sin objetos: avanzar contador
                self.frames_sin_objeto += 1

                if self.frames_sin_objeto >= self.FRAMES_VACIA_MIN:
                    # Escena confirmada vacía → iniciar cronómetro
                    if self.t_escena_vacia is None:
                        self.t_escena_vacia = time.time()

                    elapsed  = time.time() - self.t_escena_vacia
                    progreso = min(1.0, elapsed / TIEMPO_REINICIO_SEG)
                    restante = max(0.0, TIEMPO_REINICIO_SEG - elapsed)

                    self.barra_progreso.set(progreso)
                    self.lbl_reinicio_estado.configure(
                        text=f'✅ Escena vacía — reiniciando memoria en {restante:.1f}s…',
                        text_color='#2ecc71'
                    )

                    if elapsed >= TIEMPO_REINICIO_SEG:
                        self._ejecutar_reinicio()
                else:
                    self.lbl_reinicio_estado.configure(
                        text='🔄 Esperando escena vacía…',
                        text_color='#f39c12'
                    )

        elif self.modo_reinicio == REINICIO_LISTO:
            # Fase de "listo": mostrar mensaje unos segundos y volver a normal
            pass  # el after() de _ejecutar_reinicio lo maneja

    def _ejecutar_reinicio(self):
        """
        La escena estuvo vacía los segundos requeridos.
        → Borra SOLO las huellas de memoria (no el historial visual).
        → Resetea cooldowns para que el primer objeto visto sea aceptado.
        → El objeto que sea presentado a continuación se registrará como nuevo
          aunque tenga la misma apariencia que uno ya registrado.
        """
        self.modo_reinicio = REINICIO_LISTO
        self.barra_progreso.set(1.0)

        # ── Borrar memoria de huellas ────────────────────────────────────────
        self.huellas_memoria = {'Rojo': [], 'Verde': [], 'Azul': []}
        self.ultimo_intento  = {}

        # ── Reiniciar trackers para nuevo ciclo de calibración ───────────────
        self.temporizadores = {'Rojo': 0.0, 'Verde': 0.0, 'Azul': 0.0}
        for tr in self.trackers.values():
            tr.reiniciar()

        # ── Agregar separador visual en el historial ─────────────────────────
        self.after(0, self._agregar_separador_reinicio)

        self.btn_reinicio.configure(
            text='🔄 Reinicio',
            fg_color='#8e44ad', hover_color='#6c3483'
        )
        self.lbl_reinicio_estado.configure(
            text='🟢 Memoria reseteada — listo para re-detectar objetos similares',
            text_color='#2ecc71'
        )

        # Volver a estado inactivo después de 3 s
        def volver_normal():
            self.modo_reinicio = REINICIO_INACTIVO
            self.barra_progreso.set(0)
            self.lbl_reinicio_estado.configure(text='')

        self.after(3000, volver_normal)

    def _agregar_separador_reinicio(self):
        """Inserta una línea divisoria en el historial indicando el reinicio."""
        hora = datetime.datetime.now().strftime('%H:%M:%S')
        sep  = ctk.CTkFrame(
            self.lista_scroll, fg_color='#2d1f4a',
            corner_radius=6, height=28
        )
        sep.pack(fill='x', pady=5, padx=2)
        sep.pack_propagate(False)
        ctk.CTkLabel(
            sep,
            text=f'── 🔄  REINICIO  {hora} ──',
            font=('Helvetica', 9, 'bold'),
            text_color='#9b59b6'
        ).pack(expand=True)
        self.lista_scroll._parent_canvas.yview_moveto(1.0)

    def _actualizar_barra_reinicio(self):
        """Pequeña animación pulsante en la barra mientras espera objetos."""
        if self.modo_reinicio == REINICIO_ESPERANDO and self.t_escena_vacia is None:
            # Barra pulsante
            v = (time.time() % 1.0)
            self.barra_progreso.configure(progress_color='#8e44ad')
            self.barra_progreso.set(v)
            self.after(80, self._actualizar_barra_reinicio)
        else:
            self.barra_progreso.configure(progress_color='#f39c12')

    # =========================================================================
    # LÓGICA DE MEMORIA Y REGISTRO
    # =========================================================================

    COLORES_UI = {
        'Rojo':  ('#ff4d4d', '🔴'),
        'Verde': ('#4dff88', '🟢'),
        'Azul':  ('#4d9fff', '🔵'),
    }

    def on_objeto_detectado(self, nombre, orient, x_norm, y_norm, z_est, huella):
        ahora  = time.time()
        if ahora - self.ultimo_intento.get(nombre, 0.0) < self.COOLDOWN_INTENTO:
            return
        self.ultimo_intento[nombre] = ahora

        if es_mismo_objeto(huella, self.huellas_memoria[nombre]):
            return  # ya registrado → ignorar

        if huella is not None:
            self.huellas_memoria[nombre].append(huella)

        num_color     = sum(1 for o in self.historial_objetos if o['color'] == nombre) + 1
        total_memoria = sum(len(v) for v in self.huellas_memoria.values())

        entrada = {
            'color':       nombre,
            'num':         num_color,
            'orientacion': orient,
            'x':           x_norm,
            'y':           y_norm,
            'z':           z_est,
            'hora':        datetime.datetime.now().strftime('%H:%M:%S'),
        }
        self.historial_objetos.append(entrada)
        self.after(0, lambda e=entrada, tm=total_memoria: self._agregar_fila(e, tm))

    def _agregar_fila(self, entrada, total_memoria):
        color_hex, emoji = self.COLORES_UI.get(entrada['color'], ('#aaaaaa', '⚪'))

        fila = ctk.CTkFrame(
            self.lista_scroll, fg_color='#242b42',
            corner_radius=8, border_width=1, border_color='#3a4460'
        )
        fila.pack(fill='x', pady=3, padx=2)

        barra = ctk.CTkFrame(fila, fg_color=color_hex, width=5, corner_radius=4)
        barra.pack(side='left', fill='y', padx=(3, 6), pady=4)

        cont = ctk.CTkFrame(fila, fg_color='transparent')
        cont.pack(side='left', fill='both', expand=True, pady=4)

        ctk.CTkLabel(
            cont, text=f'{emoji} {entrada["color"]} #{entrada["num"]}',
            font=('Helvetica', 12, 'bold'), text_color=color_hex, anchor='w'
        ).pack(anchor='w')

        ic = '↕' if entrada['orientacion'] == 'Vertical' else ('↔' if entrada['orientacion'] == 'Horizontal' else '⊙')
        ctk.CTkLabel(
            cont, text=f'{ic} {entrada["orientacion"]}',
            font=('Helvetica', 10), text_color='#9aaac0', anchor='w'
        ).pack(anchor='w')

        ctk.CTkLabel(
            cont,
            text=f'X:{entrada["x"]:+.2f}  Y:{entrada["y"]:+.2f}  Z:{entrada["z"]}m',
            font=('Courier', 9), text_color='#6a7a9a', anchor='w'
        ).pack(anchor='w')

        ctk.CTkLabel(
            cont, text=f'🕐 {entrada["hora"]}',
            font=('Helvetica', 9), text_color='#4a5a72', anchor='w'
        ).pack(anchor='w')

        self.lbl_conteo.configure(
            text=f'Total: {len(self.historial_objetos)}  |  En memoria: {total_memoria}'
        )
        self.lista_scroll._parent_canvas.yview_moveto(1.0)

    def _limpiar_todo(self):
        """Borra lista visual + memoria (objetos visibles se re-detectan)."""
        for w in self.lista_scroll.winfo_children():
            w.destroy()
        self.historial_objetos.clear()
        self.huellas_memoria   = {'Rojo': [], 'Verde': [], 'Azul': []}
        self.ultimo_intento    = {}
        self.temporizadores    = {'Rojo': 0.0, 'Verde': 0.0, 'Azul': 0.0}
        for tr in self.trackers.values():
            tr.reiniciar()
        self.lbl_conteo.configure(text='Total: 0  |  En memoria: 0')

    # =========================================================================
    # LÓGICA DE LA APP
    # =========================================================================

    def _accion_buscar(self):
        self.btn_detectar.configure(text='Buscando hardware...', state='disabled')
        self.update()
        for w in self.frame_lista.winfo_children():
            w.destroy()
        camaras = detectar_camaras_sistema()
        if camaras:
            self.lbl_estado.configure(
                text=f'Se encontraron {len(camaras)} cámara(s)', text_color='#2ecc71'
            )
            for idx, etiqueta in camaras:
                ctk.CTkRadioButton(
                    self.frame_lista, text=etiqueta,
                    variable=self.indice_sel, value=str(idx),
                    font=('Helvetica', 13)
                ).pack(anchor='w', pady=5, padx=10)
            self.indice_sel.set(str(camaras[0][0]))
            self.lbl_pie.configure(text='Selecciona la cámara y presiona Iniciar')
            self.btn_iniciar.configure(state='normal')
        else:
            self.lbl_estado.configure(
                text='No se detectó hardware de video', text_color='#e74c3c'
            )
            self.lbl_pie.configure(text='Asegúrate de que la cámara esté conectada')
            self.btn_iniciar.configure(state='disabled')
        self.btn_detectar.configure(text='🔍 DETECTAR CÁMARAS', state='normal')

    def _iniciar_camara(self):
        indice = int(self.indice_sel.get())
        if indice == -1:
            return
        self.btn_detectar.pack_forget()
        self.lbl_estado.pack_forget()
        self.lbl_lista_tit.pack_forget()
        self.frame_lista.pack_forget()
        self.lbl_pie.pack_forget()
        self.btn_iniciar.pack_forget()
        self.lbl_titulo.configure(text='MONITOR EN VIVO  —  V0.2')
        self.frame_monitor.pack(expand=True, fill='both')
        self.cap = cv2.VideoCapture(indice, cv2.CAP_DSHOW)
        self.escaneando = False
        self.btn_escaneo.configure(text='▶ Iniciar Escaneo', fg_color='#f39c12', hover_color='#d68910')
        self._loop_video()

    def _toggle_escaneo(self):
        self.escaneando = not self.escaneando
        if self.escaneando:
            self.btn_escaneo.configure(text='⏸ Pausar Escaneo', fg_color='#d35400', hover_color='#a04000')
            self.btn_reinicio.configure(state='normal')
            self.temporizadores = {'Rojo': 0.0, 'Verde': 0.0, 'Azul': 0.0}
            for tr in self.trackers.values():
                tr.reiniciar()
        else:
            self.btn_escaneo.configure(text='▶ Reanudar Escaneo', fg_color='#f39c12', hover_color='#d68910')
            self.btn_reinicio.configure(state='disabled')
            if self.modo_reinicio != REINICIO_INACTIVO:
                self._cancelar_reinicio()

    def _loop_video(self):
        if not (self.cap and self.cap.isOpened()):
            return
        ret, frame = self.cap.read()
        if ret:
            frame = cv2.flip(frame, 1)
            hay_objetos = False

            if self.escaneando:
                frame, hay_objetos = procesar_frame_vision(
                    frame,
                    self.temporizadores,
                    self.trackers,
                    callback_objeto=self.on_objeto_detectado
                )
                # ── Actualizar máquina de estados del Reinicio ────────────────
                self._procesar_estado_reinicio(hay_objetos)

            img = ctk.CTkImage(
                light_image=Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)),
                size=(920, 630)
            )
            self.lbl_video.configure(image=img)
        self.after(15, self._loop_video)

    def _apagar_hardware(self):
        if self.cap:
            self.cap.release()
            self.cap = None
        if self.modo_reinicio != REINICIO_INACTIVO:
            self._cancelar_reinicio()
        self.escaneando = False
        self.frame_monitor.pack_forget()
        self.lbl_titulo.configure(text='CONFIGURACIÓN DE CÁMARA')
        self.btn_detectar.pack(pady=10)
        self.lbl_estado.pack(pady=(0, 20))
        self.lbl_lista_tit.pack(anchor='w', padx=150)
        self.frame_lista.pack(pady=5, padx=150)
        self.lbl_pie.pack(anchor='w', padx=150)
        self.btn_iniciar.pack(pady=(40, 0))

    def destroy(self):
        self._apagar_hardware()
        super().destroy()


print('Celda 3 V0.2: UI + Reinicio inteligente lista.')

Celda 3 V0.2: UI + Reinicio inteligente lista.


In [4]:
# Celda 4 – Punto de entrada  [V0.2]
if __name__ == '__main__':
    app = EscanerApp()
    app.mainloop()